# Demo 4 — Collegare il piano z alla risposta in frequenza

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-3485c94980f2f6df")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        url = str(location.origin) + prefisso + "/_jupyterlite/files/audiolab-course.zip?v=3485c94980f2f6df"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 44100
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Piano z", "Guadagno"])
angoli = np.linspace(0, 2*np.pi, 400)
fig.add_trace(go.Scatter(x=np.cos(angoli), y=np.sin(angoli),
    mode="lines", line=dict(color="gray", dash="dot"),
    name="Cerchio unitario"), row=1, col=1)
for nome, b_z, a_z in (("Media FIR", [0.5, 0.5], [1., 0.]),
                       ("Media IIR", [0.1, 0.], [1., -0.9])):
    zeri, poli, _ = signal.tf2zpk(b_z, a_z)
    for valori, simbolo, tipo in ((zeri, "circle-open", "zeri"), (poli, "x", "poli")):
        fig.add_trace(go.Scatter(x=valori.real, y=valori.imag, mode="markers",
            marker=dict(symbol=simbolo, size=12), name=f"{nome}: {tipo}"), row=1, col=1)
    f_z, H_z = signal.freqz(b_z, a_z, worN=2048, fs=SR)
    fig.add_trace(go.Scatter(x=f_z, y=al.to_db(np.abs(H_z), floor=1e-6), name=nome), row=1, col=2)
fig.update_xaxes(title_text="Parte reale", range=[-1.2, 1.2], row=1, col=1)
fig.update_yaxes(title_text="Parte immaginaria", range=[-1.2, 1.2],
                 scaleanchor="x", scaleratio=1, row=1, col=1)
fig.update_xaxes(title_text="Frequenza [Hz]", row=1, col=2)
fig.update_yaxes(title_text="Guadagno [dB]", range=[-60, 3], row=1, col=2)
fig.update_layout(height=430)
fig.show()
